# LogiGate — Fine-tuning Motor de Daños en Colab (GPU T4)

Reentrena el detector YOLOv8-seg en la nube. La T4 de Colab (16GB) es mucho más potente que la GTX 1650 local (4GB).

**Antes de correr:** `Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)`.

Al final descargas `best.pt`, lo renombras a `damage_model_custom.pt` y lo pones en `logigate-backend/`, luego reinicias el backend.

## 1. Verificar GPU

In [ ]:
!nvidia-smi

## 2. Instalar dependencias

In [ ]:
!pip install -q ultralytics roboflow

## 3. Configuración

Tus datos de Roboflow. Si subiste imágenes nuevas (ej. colisiones frontales severas) y generaste una versión nueva, sube `VERSION`.

In [ ]:
ROBOFLOW_API_KEY = "7P3l47cTE5Zr3RRA5zuG"
WORKSPACE        = "car-damaged-detection-e66m0"
PROJECT          = "car-damaged-severity-detection"
VERSION          = 29

DATASET_DIR = "/content/damage_dataset"
OUTPUT_DIR  = "/content/damage_model_trained"

# Hiperparámetros tuneados para T4 (16GB) — más batch e imgsz que en local
EPOCHS   = 100
IMGSZ    = 640
BATCH    = 16
PATIENCE = 25

# Fine-tune desde un modelo seg pre-entrenado. Si quieres partir de tu .pt
# actual, súbelo a /content/ y pon BASE_MODEL = '/content/damage_model_custom.pt'
BASE_MODEL = "yolov8s-seg.pt"

## 4. Descargar dataset de Roboflow

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
proj = rf.workspace(WORKSPACE).project(PROJECT)
dataset = proj.version(VERSION).download("yolov8", location=DATASET_DIR, overwrite=True)
print("Dataset en:", dataset.location)

import glob, os
yaml_path = glob.glob(os.path.join(dataset.location, "**", "data.yaml"), recursive=True)[0]
print("data.yaml:", yaml_path)
print(open(yaml_path).read())

## 5. Entrenar

~100 épocas en T4 con este dataset: aprox. 1–2 h (corta antes si `patience` se dispara). Mantén la pestaña viva.

In [ ]:
from ultralytics import YOLO

model = YOLO(BASE_MODEL)
results = model.train(
    data=yaml_path,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=PATIENCE,
    device=0,
    project=OUTPUT_DIR,
    name="damage_v2",
    exist_ok=True,
    # Augmentaciones para mejor generalización
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    flipud=0.1,
    fliplr=0.5,
    mosaic=1.0,
    copy_paste=0.3,
)

## 6. Ver métricas y validar

In [ ]:
best = f"{OUTPUT_DIR}/damage_v2/weights/best.pt"
print("Mejor modelo:", best)

# Validación sobre el split de test
m = YOLO(best)
metrics = m.val(data=yaml_path, split="test", imgsz=IMGSZ)
print("mAP50:", metrics.box.map50, " mAP50-95:", metrics.box.map)

# Curvas y matriz de confusión
from IPython.display import Image as IPImage, display
import os
rdir = f"{OUTPUT_DIR}/damage_v2"
for f in ["results.png", "confusion_matrix.png", "PR_curve.png"]:
    p = os.path.join(rdir, f)
    if os.path.exists(p):
        print(f)
        display(IPImage(p))

## 7. Probar con una imagen tuya (opcional)

Sube una captura real (ej. la colisión frontal severa) y verifica que ahora detecte mejor.

In [ ]:
from google.colab import files
up = files.upload()
for fname in up:
    res = m(fname, conf=0.25, iou=0.45)
    for r in res:
        print(fname, "->", [(m.names[int(b.cls[0])], round(float(b.conf[0]), 2)) for b in r.boxes])
        r.save(filename=f"pred_{fname}")
        display(IPImage(f"pred_{fname}"))

## 8. Descargar el modelo entrenado

Renombra el archivo descargado a `damage_model_custom.pt`, pásalo a `logigate-backend/` (reemplazando el viejo) y reinicia el backend.

In [ ]:
from google.colab import files
files.download(best)